In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2026 Johnny Nuñez Cano
# SPDX-License-Identifier: MIT
# Adapted from https://github.com/johnnynunez/blogs/tree/main/Article_2
# See ../LICENSE and ../THIRD_PARTY_NOTICES.md for attribution.


# Build the Pick and Place


In the previous notebook you explored MuJoCo's building blocks one at a time. Now you will assemble them into a complete, runnable robot program.

The file `so101_pick_place.py` has been prepared with six exercises marked `TODO Step 0` through `TODO Step 5`. Your job is to fill them in. When you are finished, the SO-101 will pick up the red cube and stack it on the blue one.

## Outline

- Confirm the starting file does not yet run.
- Complete six `TODO Step` exercises in `so101_pick_place.py`.
- Run the finished task headless and verify the stack.
- Optionally watch it in the interactive viewer.

---

## What you will learn

By the end of this part, you will be able to:

- Write a complete MuJoCo simulation loop from scratch.
- Separate control rate from physics rate using substeps.
- Verify a manipulation task programmatically rather than by eye.

## Setup

In a local checkout, install the packages using the [tutorial README](../README.md) and select that environment as the notebook kernel. The cell below locates the checkout and changes into this lesson's directory so that the exercise scripts and imports resolve correctly.

In Google Colab, the cell downloads this tutorial from the repository's `main` branch and installs its pinned dependencies. For the GPU lesson, select **Runtime → Change runtime type → T4 GPU** (or another available NVIDIA GPU) first. If Colab requests a runtime restart after installation, restart and rerun the setup cell.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab
    in_colab = True
except ImportError:
    in_colab = False

lesson_relative = Path("tutorials/warp/notebooks/mujoco/part1")
repo_root = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / lesson_relative / "pick_place_common.py").is_file()),
    None,
)
if repo_root is None and in_colab:
    repo_root = Path("/content/accelerated-computing-hub")
    if not repo_root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
             "--branch", "main",
             "https://github.com/NVIDIA/accelerated-computing-hub.git",
             str(repo_root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
        subprocess.run(
            ["git", "-C", str(repo_root), "sparse-checkout", "set",
             "tutorials/warp/notebooks/mujoco"],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
if repo_root is None or not (repo_root / lesson_relative / "pick_place_common.py").is_file():
    raise FileNotFoundError("Open this notebook from an accelerated-computing-hub checkout.")

if in_colab:
    # Colab has no desktop display; use its GPU-backed EGL context.
    os.environ.setdefault("MUJOCO_GL", "egl")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(repo_root / "tutorials/warp/notebooks/mujoco/requirements.txt")],
        check=True,
    )
lesson_dir = repo_root / lesson_relative
os.chdir(lesson_dir)
if str(lesson_dir) not in sys.path:
    sys.path.insert(0, str(lesson_dir))
print("Lesson directory:", lesson_dir)


## Overview of the files

Open `so101_pick_place.py` in your editor and keep it visible alongside this notebook as you work through the steps.

If at any point you want to compare against a reference, each step has a snippet in `solutions/step_NN_*.py`, and the fully completed program is `solutions/so101_pick_place_solution.py`.

First, let's make a backup of the original so you can always start over.

In [ ]:
import shutil

backup = Path("so101_pick_place_original.py")
if not backup.exists():
    shutil.copyfile("so101_pick_place.py", backup)
print("Backup:", backup)


## Baseline: confirm it does not run yet

The starting file is intentionally incomplete. Run it now and read the error — you should see an `AttributeError`, because `model` is still `None`.

In [ ]:
!{sys.executable} so101_pick_place.py --headless-steps 10

### 0. Import MuJoCo

The file imports `numpy` but not the physics engine. At `TODO Step 0`, import both the core module and the viewer submodule.

`mujoco.viewer` is a separate submodule, so `import mujoco` alone will not give you `mujoco.viewer.launch_passive`.

### 1. Compile the model and allocate state

At `TODO Step 1`, replace the two `None` placeholders:

- Build the model with `load_pick_place_model(xml_path, spec)`. The helper applies the SO-101 arm force-limit override of **±30 N·m** rather than the asset's ±2.94 N·m. This is a simulation convenience, not a physical hardware rating; the gripper limit is unchanged.
- Allocate state with `mujoco.MjData(model)`.

The script then sets the timestep to `1 / fps / sim_substeps`: with the defaults, this is **0.002 s**. Treat the configured model as shared during simulation; each world owns separate state.


### 2. Set the initial conditions

At `TODO Step 2`, initialize the arm and props using the resolved robot profile:

- `apply_arm_ctrl(model, data, spec.home_ctrl)` sets the home pose and runs forward kinematics.
- `reset_cubes(model, data, spec)` sets the cubes' starting poses and zero velocity.

Setting state directly is appropriate for initialization. During the task, evolve the live state through `mj_step`.


### 3. Create the controller

At `TODO Step 3`, instantiate `PickPlaceController()`.

It owns the waypoint state machine, damped-least-squares IK, and gradual gripper close. This is an open-loop scripted controller: it computes joint targets using scratch kinematic state and previous commanded targets, rather than correcting waypoints from observed cube or arm state. The GPU demonstration reuses it on the CPU and mirrors simulated state back for visualization and task checks.


### 4. Step the physics

At `TODO Step 4`, repeat these operations for each of the `sim_substeps` iterations:

1. Copy the controller output into `data.ctrl[: model.nu]`.
2. Call `mujoco.mj_step(model, data)`.

The controller runs at 50 Hz by default, holding its command over 10 physics substeps of 0.002 s. This separates the command interval from the physics integration interval.

In the GPU lesson, `mjw.step(m, d)` advances batched device state. The migration also requires allocating and transferring that state; changing the step call alone is insufficient.


### 5. Verify the stack

At `TODO Step 5`, print the final cube positions so success can be checked programmatically instead of by eye.

Call `mujoco.mj_forward(model, data)` to refresh derived body positions for the final state, then read `data.xpos[red_body]` and `data.xpos[blue_body]` and report two numbers:

- the horizontal distance between the two cube centers (should be close to zero)
- the vertical gap between them (should be roughly `2 * 0.022 = 0.044` m, one cube height)

## Run your finished program

Run the task headless for 600 frames. With `--debug` you also get per-phase telemetry showing the aperture center and both cube positions as the sequence progresses.

In [ ]:
!{sys.executable} so101_pick_place.py --headless-steps 600 --debug
# Same task on the Seeed reBot DevArm:
# !{sys.executable} so101_pick_place.py --robot rebot --headless-steps 600 --debug
# Automated acceptance checks in the completed reference:
!{sys.executable} solutions/so101_pick_place_solution.py --headless-steps 600 --test


If the stack succeeded you should see a small `xy_err` and a `dz` near 0.044 m in the final line.

If the cube ended up on the floor, the usual causes are:

- The `ctrl` vector is not being copied inside the substep loop, so the arm never tracks the target.
- `reset_cubes` was skipped, so the cubes start in the wrong place.
- The substep loop calls `mj_forward` instead of `mj_step`, which computes kinematics without advancing dynamics.

## Watch it run

The passive viewer opens a desktop window. In a local checkout, open a terminal in `tutorials/warp/notebooks/mujoco/part1` and run your completed exercise:

```bash
python so101_pick_place.py
```

To run the supplied reference immediately:

```bash
python solutions/so101_pick_place_solution.py
```

On macOS the viewer requires `mjpython`; the script attempts to relaunch itself automatically. A remote notebook without a display can use the headless commands instead.


## Verify against the reference

You now have a working robot manipulation task. To verify, compare your `so101_pick_place.py` against `solutions/so101_pick_place_solution.py`:

In [ ]:
!diff so101_pick_place.py solutions/so101_pick_place_solution.py
# The reference also adds validation helpers and an import path for solutions/.
# Focus on the completed TODO steps; the files need not be byte-identical.


## Measure fixed-control CPU physics throughput

This separate measurement holds the home control vector fixed. It excludes the waypoint controller, inverse kinematics, rendering, and resets from the timed region. It measures world-steps per second, **not completed manipulation tasks per second**.

Resolve the robot profile before reading its home pose; module-level aliases imported before scene resolution can retain their initial values. Match the GPU benchmark's 0.002 s timestep, initial state, 10 warm-up steps, and 200 measured steps.


In [ ]:
import time
import mujoco
from pick_place_common import (
    apply_arm_ctrl, load_pick_place_model, reset_cubes, resolve_pick_place_scene,
)
from robots import get_robot

spec = get_robot("so101")
xml_path = resolve_pick_place_scene(spec=spec)
model = load_pick_place_model(xml_path, spec)
model.opt.timestep = 0.002
data = mujoco.MjData(model)
apply_arm_ctrl(model, data, spec.home_ctrl)
reset_cubes(model, data, spec)

for _ in range(10):
    mujoco.mj_step(model, data)

n_steps = 200
t0 = time.perf_counter()
for _ in range(n_steps):
    mujoco.mj_step(model, data)
elapsed = time.perf_counter() - t0

baseline = n_steps / elapsed
print(f"Fixed-control CPU physics: {baseline:,.0f} world-steps/second (1 world)")
print(f"Step latency: {elapsed / n_steps * 1e6:.1f} microseconds")
print("Excludes controller, rendering, resets, and task-success evaluation.")


A sequential Python loop over many CPU worlds would add their step costs, but this is not a comparison against all CPU parallelism. [`mujoco.rollout`](https://mujoco.readthedocs.io/en/stable/python.html#rollout) supports multithreaded CPU execution and should be included when evaluating a CPU batch alternative.

Repeat measurements and record hardware, package versions, solver settings, precision, timestep, contact state, and batch size before reporting a speedup. The CPU implementation uses double precision; this tutorial's MJWarp state uses `float32`, so the comparison is not a numerically identical computation.

---

## Next

Continue to [Scaling with MuJoCo Warp](../part2/07__mujoco_warp.ipynb), where you upload the model, allocate batched states, and measure a GPU scaling curve.
